# 맨홀-노드 STGNN 실험 v1 — 관측 노드 loss

`09_manhole_graph_flow_corrected.ipynb`가 만든 맨홀 그래프 산출물을 실제 GNN 학습 입력으로 연결한다.

- 그래프: 노드=맨홀, 엣지=관저고 기반 방향 보정 유향 관거
- 입력: 현재 보유한 하수 수위 센서와 AWS 점강우를 snap된 맨홀 노드에 주입
- 학습/평가: 데이터가 있는 관측 맨홀 노드에서만 loss와 CSI 계산
- 미관측 맨홀은 관망 message passing 경유 노드로 남기되, 정답에는 포함하지 않는다.

> 실행 전제: 먼저 `09_manhole_graph_flow_corrected.ipynb`를 실행해 `gnn_manhole_*` 산출물을 생성해야 한다.


In [ ]:
import os
os.chdir('/home/namjun/city_flood')
import sys
sys.path.insert(0, 'scripts')
from krfont import set_korean
set_korean()

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

EB = Path('dataset/processed/eda_based')
THR = 0.5
W = 6
HZ = [1, 3, 6]
SEEDS = [0, 1, 2]
BATCH = 16
EPOCHS = 20

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', dev)


## 1. 맨홀 그래프 산출물 로드

`09`의 산출물이 없으면 여기서 멈춘다. 이 노트북은 원 GIS를 다시 읽지 않고, 이미 생성된 그래프 파일만 사용한다.


In [ ]:
required = [
    EB / 'gnn_manhole_nodes.parquet',
    EB / 'gnn_manhole_edges.parquet',
    EB / 'gnn_manhole_edge_index.npy',
    EB / 'gnn_manhole_sensor_snap.parquet',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    msg = '먼저 09_manhole_graph_flow_corrected.ipynb를 실행해 산출물을 생성하세요:\n' + '\n'.join(missing)
    raise FileNotFoundError(msg)

nodes = pd.read_parquet(EB / 'gnn_manhole_nodes.parquet')
edges = pd.read_parquet(EB / 'gnn_manhole_edges.parquet')
snap = pd.read_parquet(EB / 'gnn_manhole_sensor_snap.parquet')
edge_index_np = np.load(EB / 'gnn_manhole_edge_index.npy')

nodes['node_id'] = nodes.node_id.astype(str)
nodes = nodes.sort_values('node_idx').reset_index(drop=True)
assert (nodes.node_idx.to_numpy() == np.arange(len(nodes))).all(), 'node_idx가 0..N-1 순서가 아닙니다.'
N = len(nodes)
edge_index = torch.as_tensor(edge_index_np, dtype=torch.long, device=dev)

snap['node_id'] = snap.node_id.astype(str)
if 'node_idx' not in snap.columns:
    snap = snap.merge(nodes[['node_id', 'node_idx']], on='node_id', how='left', validate='many_to_one')
if snap.node_idx.isna().any():
    bad = snap.loc[snap.node_idx.isna(), ['sensor_id', 'node_id']].head()
    raise ValueError(f'sensor_snap의 node_id가 nodes에 매칭되지 않습니다:\n{bad}')

observed_snap = snap[snap.is_observed == True].copy()
observed_snap['sensor_id'] = observed_snap['sensor_id'].astype(str)
observed_snap['node_idx'] = observed_snap.node_idx.astype(int)
observed_node_idx = observed_snap.node_idx.to_numpy()
observed_mask = np.zeros(N, dtype=bool)
observed_mask[observed_node_idx] = True

print(f'노드 {N:,} | 엣지 {edge_index.shape[1]:,} | 관측 센서 {len(observed_snap):,} | 관측 노드 {observed_mask.sum():,}')
print('동일 맨홀에 여러 관측 센서가 snap된 경우:', int(len(observed_snap) - observed_mask.sum()))


## 2. 시계열을 맨홀 노드 텐서로 정렬

현재 보유 데이터는 모든 맨홀의 수위가 아니라 일부 센서 수위뿐이다. 따라서 관측 센서가 snap된 맨홀에만 `fill_rate`, `trend`, AWS 점강우를 넣고, 나머지 맨홀은 0과 관측마스크 피처로 둔다. 이 구조는 이후 레이더 강우·DEM/SWMM 유입량이 들어오면 그대로 확장할 수 있다.


In [ ]:
obs_nodes = observed_snap.node_idx.astype(int).to_numpy()
sensor_ids = observed_snap.sensor_id.astype(str).tolist()
node_order = pd.Series(obs_nodes, index=sensor_ids)

sf = pd.read_parquet(
    EB / 'sewer_features_10min.parquet',
    columns=['sewer_sensor_id', 'ts10', 'fill_rate'],
)
sf['sewer_sensor_id'] = sf.sewer_sensor_id.astype(str)
sf = sf[sf.sewer_sensor_id.isin(sensor_ids)].copy()
sf['ts10'] = pd.to_datetime(sf.ts10)
Wf_sensor = sf.pivot_table(index='ts10', columns='sewer_sensor_id', values='fill_rate').reindex(columns=sensor_ids)

m = pd.read_parquet(EB / 'aws_sewer_mapping_v2.parquet', columns=['sensor_id', 'aws_stn'])
m['sensor_id'] = m.sensor_id.astype(str)
r = pd.read_parquet('data/aws_seoul_rain_10min.parquet', columns=['stn', 'ts10', 'rn60m']).sort_values(['stn', 'ts10'])
r['ts10'] = pd.to_datetime(r.ts10)
r['rain6h'] = r.groupby('stn')['rn60m'].transform(lambda s: s.fillna(0).rolling(36, min_periods=1).max())

start = max(pd.Timestamp('2024-06-01'), Wf_sensor.index.min())
grid = pd.date_range(start, Wf_sensor.index.max(), freq='10min')
Wf_sensor = Wf_sensor.reindex(grid).sort_index()
T = len(grid)
O = len(sensor_ids)

fill_obs = Wf_sensor.to_numpy(dtype=np.float32)
valid_obs = ~np.isnan(fill_obs)
fill_ff_obs = pd.DataFrame(fill_obs).ffill().fillna(0).to_numpy(dtype=np.float32)
trend_obs = fill_ff_obs - np.vstack([fill_ff_obs[:1], fill_ff_obs[:-1]])
rn_obs = np.zeros((T, O), np.float32)
r6_obs = np.zeros((T, O), np.float32)

for j, sid in enumerate(sensor_ids):
    mm = m[m.sensor_id == sid]
    if not mm.empty:
        st = int(mm.aws_stn.iloc[0])
        rr = r[r.stn == st].set_index('ts10').reindex(grid)
        rn_obs[:, j] = rr.rn60m.fillna(0).to_numpy(dtype=np.float32)
        r6_obs[:, j] = rr.rain6h.fillna(0).to_numpy(dtype=np.float32)

hour = grid.hour.to_numpy()
hs = np.sin(2 * np.pi * hour / 24).astype(np.float32)
hc = np.cos(2 * np.pi * hour / 24).astype(np.float32)
obs_feat = observed_mask.astype(np.float32)

Fdim = 7
maxh = max(HZ)
starts = np.arange(W - 1, T - maxh)
ts = grid[starts]
tr = ts < pd.Timestamp('2025-05-01')
train_idx = np.flatnonzero(tr)
test_idx = np.flatnonzero(~tr)

future = starts[:, None] + np.asarray(HZ)[None, :]
Yobs = (fill_obs[future] >= THR).transpose(0, 2, 1).astype(np.float32)
Vobs = (valid_obs[future] & valid_obs[starts][:, None, :]).transpose(0, 2, 1)

print('샘플', len(starts), 'train', len(train_idx), 'test', len(test_idx))
print('관측 노드', O, '| compact loss 위치:', int(Vobs.sum()), '| test 양성:', [int(Yobs[test_idx, :, h][Vobs[test_idx, :, h]].sum()) for h in range(len(HZ))])


## 3. 유향 메시지 패싱 STGNN

기존 `06`의 dense 인접행렬 대신 `edge_index`를 사용한다. 큰 13k×13k 행렬을 만들지 않고, `src→dst` 엣지를 따라 상류 노드 은닉상태를 하류 노드에 집계한다.


In [ ]:
def make_batch(sample_idx):
    sample_idx = np.asarray(sample_idx)
    s = starts[sample_idx]
    hist = s[:, None] - np.arange(W - 1, -1, -1)[None, :]
    b = len(sample_idx)

    xb = np.zeros((b, N, W, Fdim), dtype=np.float32)
    # Numpy advanced indexing places the obs-node axis first: (O, B, W).
    xb[:, obs_nodes, :, 0] = fill_ff_obs[hist].transpose(2, 0, 1)
    xb[:, obs_nodes, :, 1] = trend_obs[hist].transpose(2, 0, 1)
    xb[:, obs_nodes, :, 2] = rn_obs[hist].transpose(2, 0, 1)
    xb[:, obs_nodes, :, 3] = r6_obs[hist].transpose(2, 0, 1)
    xb[:, :, :, 4] = obs_feat[None, :, None]
    xb[:, :, :, 5] = hs[hist][:, None, :]
    xb[:, :, :, 6] = hc[hist][:, None, :]

    yb = np.zeros((b, N, len(HZ)), dtype=np.float32)
    vb = np.zeros((b, N, len(HZ)), dtype=bool)
    yb[:, obs_nodes, :] = Yobs[sample_idx]
    vb[:, obs_nodes, :] = Vobs[sample_idx]
    return torch.as_tensor(xb, device=dev), torch.as_tensor(yb, device=dev), torch.as_tensor(vb, device=dev)


class ManholeSTGNN(nn.Module):
    def __init__(self, fdim, hid=24, graph=True):
        super().__init__()
        self.graph = graph
        self.gru = nn.GRU(fdim, hid, batch_first=True)
        self.self_lin = nn.Linear(hid, hid)
        self.nbr_lin = nn.Linear(hid, hid)
        self.head = nn.Sequential(nn.ReLU(), nn.Linear(hid, len(HZ)))

    def aggregate_upstream(self, h):
        src, dst = edge_index
        msg = h[:, src, :]
        agg = torch.zeros_like(h)
        agg.index_add_(1, dst, msg)
        deg = torch.zeros(h.shape[1], device=h.device, dtype=h.dtype)
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=h.dtype))
        return agg / deg.clamp_min(1).view(1, -1, 1)

    def forward(self, x):
        b, n, wl, fd = x.shape
        h, _ = self.gru(x.reshape(b * n, wl, fd))
        h = h[:, -1].reshape(b, n, -1)
        if self.graph:
            z = torch.relu(self.self_lin(h) + self.nbr_lin(self.aggregate_upstream(h)))
        else:
            z = torch.relu(self.self_lin(h))
        return self.head(z)


def csi(yt, pt):
    tp = (yt & pt).sum()
    fp = ((~yt) & pt).sum()
    fn = (yt & (~pt)).sum()
    return tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0


def predict_observed(net, sample_indices, bs=BATCH):
    preds = []
    net.eval()
    with torch.no_grad():
        for i in range(0, len(sample_indices), bs):
            xb, _, _ = make_batch(sample_indices[i:i + bs])
            out = torch.sigmoid(net(xb))[:, obs_nodes, :].cpu().numpy()
            preds.append(out)
    return np.concatenate(preds, axis=0)


def train_eval(graph, seed, epochs=EPOCHS, bs=BATCH):
    torch.manual_seed(seed)
    net = ManholeSTGNN(Fdim, graph=graph).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    pos = Yobs[train_idx][Vobs[train_idx]].sum()
    neg = Vobs[train_idx].sum() - pos
    pos_weight = torch.tensor(np.clip(neg / (pos + 1e-6), 1, 50), dtype=torch.float32, device=dev)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')

    for ep in range(epochs):
        net.train()
        perm = np.random.default_rng(seed + ep).permutation(train_idx)
        for i in range(0, len(perm), bs):
            xb, yb, vb = make_batch(perm[i:i + bs])
            opt.zero_grad()
            out = net(xb)
            loss = (loss_fn(out, yb) * vb).sum() / (vb.sum() + 1e-6)
            loss.backward()
            opt.step()

    pr_train = predict_observed(net, train_idx, bs=bs)
    pr_test = predict_observed(net, test_idx, bs=bs)
    return pr_train, pr_test


def tune_threshold(pr_train, hi):
    ytr = Yobs[train_idx, :, hi].astype(bool)[Vobs[train_idx, :, hi]]
    ptr = pr_train[:, :, hi][Vobs[train_idx, :, hi]]
    best, bp = -1, 0.5
    for c in np.linspace(0.05, 0.95, 19):
        score = csi(ytr, ptr >= c)
        if score > best:
            best, bp = score, c
    return bp


def eval_csi(pr_train, pr_test):
    out = []
    for hi in range(len(HZ)):
        yte = Yobs[test_idx, :, hi].astype(bool)
        vte = Vobs[test_idx, :, hi]
        out.append(csi(yte[vte], pr_test[:, :, hi][vte] >= tune_threshold(pr_train, hi)))
    return out


## 4. Persistence / GNN-off / GNN-on 비교

현재는 관측 센서 노드에서만 평가되므로, 결과 해석은 “맨홀 전체 예측 성능”이 아니라 “관망 그래프를 넣었을 때 관측 노드 예측이 개선되는지”에 한정한다.


In [ ]:
persist = []
for hi in range(len(HZ)):
    yte = Yobs[test_idx, :, hi].astype(bool)
    vte = Vobs[test_idx, :, hi]
    base = fill_ff_obs[starts[test_idx]] >= THR
    persist.append(csi(yte[vte], base[vte]))

res = {'persistence': [persist]}

for name, use_graph in [('GNN-off(그래프X)', False), ('GNN-on(맨홀그래프)', True)]:
    vals = []
    for seed in SEEDS:
        pr_train, pr_test = train_eval(use_graph, seed)
        vals.append(eval_csi(pr_train, pr_test))
    res[name] = vals

R = pd.DataFrame({name: np.mean(vals, axis=0) for name, vals in res.items()}, index=[f'+{h * 10}분' for h in HZ])
print(R.round(3).to_string())
R.to_parquet(EB / 'gnn_manhole_stgnn_csi.parquet')


## 5. 시각화


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(HZ))
w = 0.26
cols = {'persistence': '#b0bec5', 'GNN-off(그래프X)': '#ef9a9a', 'GNN-on(맨홀그래프)': 'steelblue'}
for i, (name, color) in enumerate(cols.items()):
    label = '지속성(persistence)' if name == 'persistence' else name
    ax.bar(x + (i - 1) * w, R[name].values, w, label=label, color=color)
ax.set_xticks(x)
ax.set_xticklabels(R.index)
ax.set_ylabel('CSI (이벤트=충전율≥0.5, 관측 맨홀만)')
ax.set_title('관악 맨홀-노드 STGNN: 관측 노드 loss 평가')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
os.makedirs('reports/figures_gnn', exist_ok=True)
plt.savefig('reports/figures_gnn/11_manhole_stgnn_observed_loss.png', dpi=110, bbox_inches='tight')
plt.show()
print('saved')


## 결론 메모

- 이 노트북은 기존 센서-센서 거리 그래프가 아니라 `09`의 맨홀-노드 유향 그래프를 사용한다.
- loss와 CSI는 `gnn_manhole_sensor_snap.parquet` 기준 데이터 보유 센서가 붙은 맨홀 노드에서만 계산한다.
- 미관측 맨홀은 message passing 경유 노드로만 사용된다.
- 현재 입력은 센서 수위·AWS 점강우 기반이라, 보고서의 결론처럼 레이더 강우와 SWMM/DEM 집수면적이 들어오기 전까지는 조기예측 성능 개선이 제한적일 수 있다.
